# DA Fourier Transform Template

This notebook is a standalone wrapper around the repository's post-fit DA Fourier-transform workflow.
Point it at grouped DA fit/sample outputs that already contain extracted `m0(bT, bz)` values, then validate and run the downstream transform without rerunning the DA fit.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[3]
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_da_fourier_input_text,
    run_da_fourier_from_notebook,
    validate_da_fourier_notebook_config,
)


## User Inputs

These fields describe a downstream Fourier-transform job that consumes grouped DA fit/sample outputs.
Run the DA N-state fit first, or the normalization workflow first, so the grouped matrix-element outputs already exist under `input_root`.


In [2]:
DA_RESULTS = REPO_ROOT / "examples" / "l64c64a076_m140" / "analysis" / "3-normalize"

workflow_config = {
    # Data / metadata settings
    "title_pattern": "l64c64a076_m140_fit_pz*",
    "input_root": str(DA_RESULTS),
    "ns": 64,
    "lattice_spacing_fm": 0.076,
    "pzlist": [5, 6, 7, 8],
    "gmlist": ["T5"],
    "etalist": ["eta0"],
    "bTlist": [0],
    "component": "real",
    # Run again with component = "imag" to produce the imaginary part.
    "nstates": 2,
    "normalization_mode": "mode3",

    # Fourier-transform parameter settings
    "x_range": [-0.5, 1.5],
    "x_count": 201,
    "zstep_fm": 0.01,
    "interpolation_kind": "cubic",

    # Output settings
    "plot": True,
    "results_dir": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT",
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `input_root`: Root directory containing grouped DA fit outputs or normalized outputs.
- `title_pattern`: Same pattern used upstream so the Fourier workflow can resolve each per-`pz` dataset directory.
- `pzlist`, `gmlist`, `etalist`, `bTlist`: Batch job selectors. The workflow loops over all requested combinations.
- `component`, `nstates`: Select which grouped matrix-element outputs to transform.
- `normalization_mode`: Choose `"raw"`, `"mode1"`, `"mode2"`, or `"mode3"` explicitly. `raw` resolves original DA fit outputs; `mode1/2/3` resolve the corresponding normalized outputs.
- `ns`, `lattice_spacing_fm`: Physics/input metadata needed by the cosine transform.
- `x_range`, `x_count`: Convenient way to build a regular `x` grid. If you want a custom irregular grid, replace these with `x_values`.
- `zstep_fm`: Refined integration step in fm. The default downstream workflow uses `0.01` fm.
- `interpolation_kind`: Interpolation used before integrating on the refined `z` grid. Default is `"cubic"`.
- `plot`: Whether to also write the Fourier PDF plot.
- `results_dir`: Output directory for Fourier tables, sample files, and plots.


## Validate

This checks the configuration and resolves the effective `x_values` grid that will be used by the downstream Fourier run.


In [3]:
validated = validate_da_fourier_notebook_config(workflow_config)
print(pretty_print_config(validated))


{
  "title_pattern": "l64c64a076_m140_fit_pz*",
  "input_root": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/3-normalize",
  "ns": 64,
  "lattice_spacing_fm": 0.076,
  "pzlist": [
    5,
    6,
    7,
    8
  ],
  "gmlist": [
    "T5"
  ],
  "etalist": [
    "eta0"
  ],
  "bTlist": [
    0
  ],
  "component": "real",
  "nstates": 2,
  "normalization_mode": "mode3",
  "x_values": "[-0.5  -0.49 -0.48 -0.47 -0.46 -0.45 -0.44 -0.43 -0.42 -0.41 -0.4  -0.39\n -0.38 -0.37 -0.36 -0.35 -0.34 -0.33 -0.32 -0.31 -0.3  -0.29 -0.28 -0.27\n -0.26 -0.25 -0.24 -0.23 -0.22 -0.21 -0.2  -0.19 -0.18 -0.17 -0.16 -0.15\n -0.14 -0.13 -0.12 -0.11 -0.1  -0.09 -0.08 -0.07 -0.06 -0.05 -0.04 -0.03\n -0.02 -0.01  0.    0.01  0.02  0.03  0.04  0.05  0.06  0.07  0.08  0.09\n  0.1   0.11  0.12  0.13  0.14  0.15  0.16  0.17  0.18  0.19  0.2   0.21\n  0.22  0.23  0.24  0.25  0.26  0.27  0.28  0.29  0.3   0.31  0.32  0.33\n  0.34  0.35  0.36  0.37  0.38  0.39  0.4   0.41  0.42  0.4

## Preview Plain-Text Input

This is a human-readable snapshot of the same notebook configuration.


In [4]:
print(render_da_fourier_input_text(workflow_config))


title_pattern l64c64a076_m140_fit_pz*
input_root /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/3-normalize
ns 64
lattice_spacing_fm 0.076
pzlist 5 6 7 8
gmlist T5
etalist eta0
bTlist 0
component real
nstates 2
normalization_mode mode3
x_range -0.5 1.5
x_count 201
zstep_fm 0.01
interpolation_kind cubic
plot true
results_dir /Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT



## Run Fourier Workflow

This runs only the downstream Fourier step from existing grouped DA fit/sample outputs.


In [5]:
fourier_outputs = run_da_fourier_from_notebook(workflow_config)
fourier_outputs


[PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT/l64c64a076_m140_fit_pz5/tables/l64c64a076_m140_fit_pz5_T5_eta0_bT0_mode3_real_2state_fourier.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT/l64c64a076_m140_fit_pz5/samples/l64c64a076_m140_fit_pz5_T5_eta0_bT0_mode3_real_2state_fourier_samples.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT/l64c64a076_m140_fit_pz5/plots/l64c64a076_m140_fit_pz5_T5_eta0_bT0_mode3_real_2state_fourier.pdf'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT/l64c64a076_m140_fit_pz6/tables/l64c64a076_m140_fit_pz6_T5_eta0_bT0_mode3_real_2state_fourier.txt'),
 PosixPath('/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT/l64c64a076_m140_fit_pz6/samples/l64c64a076_m140_fit_pz6_T5_eta0_bT0_mode3_real_2state_f

## Config Snapshot

Keep this cell in the notebook so saved runs preserve the exact downstream Fourier settings that produced the outputs.


In [6]:
pretty_print_config(workflow_config)


'{\n  "title_pattern": "l64c64a076_m140_fit_pz*",\n  "input_root": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/3-normalize",\n  "ns": 64,\n  "lattice_spacing_fm": 0.076,\n  "pzlist": [\n    5,\n    6,\n    7,\n    8\n  ],\n  "gmlist": [\n    "T5"\n  ],\n  "etalist": [\n    "eta0"\n  ],\n  "bTlist": [\n    0\n  ],\n  "component": "real",\n  "nstates": 2,\n  "normalization_mode": "mode3",\n  "x_range": [\n    -0.5,\n    1.5\n  ],\n  "x_count": 201,\n  "zstep_fm": 0.01,\n  "interpolation_kind": "cubic",\n  "plot": true,\n  "results_dir": "/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/analysis/4-FT"\n}'